In [32]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, gc
import pickle
from time import time
from pathlib import Path
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
import joblib
import json
from time import time
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

In [3]:
TRAIN_PATH = Path(r'data\drive-download-20260622T101344Z-3-001\train')
VAL_PATH = Path(r'data\drive-download-20260622T101344Z-3-001\val')
TEST_PATH = Path(r'data\drive-download-20260622T101344Z-3-001\test')

X_TRAIN = Path('X_train.bin')
Y_TRAIN = Path('y_train.parquet')
X_VAL = Path('X_val.bin')
Y_VAL = Path('y_val.parquet')
X_TEST = Path('X_test.bin')
Y_TEST = Path('y_test.parquet')
TARGET = 'label'

CHKP_PATH = Path('CatBoost')

# ====================================
# LOAD DATA
# ====================================
if not os.path.exists(TRAIN_PATH / X_TRAIN) or not os.path.exists(TRAIN_PATH / Y_TRAIN):
    raise FileNotFoundError(f'Train files are missing. (Current file: {os.listdir(TRAIN_PATH)})')

elif not os.path.exists(VAL_PATH / X_VAL) or not os.path.exists(VAL_PATH / Y_VAL):
    raise FileNotFoundError(f'Validation files are missing. (Current files: {os.listdir(VAL_PATH)})')

elif not os.path.exists(TEST_PATH / X_TEST) or not os.path.exists(TEST_PATH / Y_TEST):
    raise FileNotFoundError(f'Test files are missing. (Current files: {os.listdir(TEST_PATH)})')

else:
    print('Loading Data...')
    y_train = pd.read_parquet(TRAIN_PATH / Y_TRAIN)
    y_val = pd.read_parquet(VAL_PATH / Y_VAL)
    y_test = pd.read_parquet(TEST_PATH / Y_TEST)
    
    train_rows = len(y_train)
    val_rows = len(y_val)
    test_rows = len(y_test)
    n_feats = 100

    X_train = np.memmap(
        TRAIN_PATH / X_TRAIN,
        dtype=np.float32,
        mode='r',
        shape=(train_rows, n_feats)
    )

    X_val = np.memmap(
        VAL_PATH / X_VAL,
        dtype=np.float32,
        mode='r',
        shape=(val_rows, n_feats)
    )

    X_test = np.memmap(
        TEST_PATH / X_TEST,
        dtype=np.float32,
        mode='r',
        shape=(test_rows, n_feats)
    )

    X_train = np.array(X_train)
    y_train = y_train[TARGET].values
    X_val = np.array(X_val)
    y_val = y_val[TARGET].values
    X_test = np.array(X_test)
    y_test = y_test[TARGET].values
    print('Done Loading!')

# Choosing top-k features from 100 features
top_k = 100
print(f'Take top-{top_k} features from the data')
X_train = X_train[:, :top_k]
X_val = X_val[:, :top_k]

print('Dataset Size and Unique Classes:')
print(X_train.shape, y_train.shape, X_val.shape, y_val.shape)

Loading Data...
Done Loading!
Take top-100 features from the data
Dataset Size and Unique Classes:
(2417725, 100) (2417725,) (1073847, 100) (1073847,)


In [4]:
with open('feat_idx.pkl', 'rb') as f:
    feat_idx = pickle.load(f)

with open('selected_cols.pkl', 'rb') as f:
    selected_cols = pickle.load(f)

# Convert to DataFrame
X_train_df = pd.DataFrame(X_train, columns=selected_cols)
X_val_df = pd.DataFrame(X_val, columns=selected_cols)
X_test_df = pd.DataFrame(X_test, columns=selected_cols)


### Fitting in drop-one-group training sets

In [ ]:
for group_name, ids in feat_idx.items():
    group_cols = [col for col in X_train_df.columns if col in ids]
    if len(group_cols) <= 0:
        print(f"Not exist features of {group_name} in Top-100!")
        continue
    print(f"Columns of {group_name}: {group_cols}")

    # Remove columns
    X_train_red = X_train_df.drop(columns=group_cols)
    X_val_red = X_val_df.drop(columns=group_cols)
    X_test_red = X_test_df.drop(columns=group_cols)
    print(f'Training(w/o {group_name})...')

    # ====================================
    # TRAINING WITH BEST HYPER-PARAMETERS
    # ====================================
    train_start = time()
    # CatBoost
    cb = CatBoostClassifier()
    cb.load_model(Path('CatBoost/catboost_checkpoint.cbm'))
    cb_params = cb.get_params()
    cb_params['train_dir'] = f'./catboost_info/{group_name}'
    cb_new = CatBoostClassifier(**cb_params)
    cb_new.fit(X_train_red, y_train, eval_set=(X_val_red, y_val))

    # XGBoost
    xgb = XGBClassifier()
    xgb.load_model(Path('xgboost_checkpoint.json'))
    xgb_params = xgb.get_params()
    xgb_new = XGBClassifier(**xgb_params)
    xgb_new.fit(X_train_red, y_train)

    # LGBM
    lgbm = joblib.load(Path('LGBM/lightgbm_checkpoint.joblib'))
    lgbm_params = lgbm.get_params()
    lgbm_new = LGBMClassifier(**lgbm_params)
    lgbm_new.fit(X_train_red, y_train)
    train_end = time()

    print(f'Training Time(w/o {group_name}): {train_end-train_start:.4f}')

    # ====================================
    # PREDICT PROBABILITIES
    # ====================================
    print('\nPredicting...')
    start = time()
    cb_proba = cb_new.predict_proba(X_val_red)[:, 1]
    xgb_proba = xgb_new.predict_proba(X_val_red)[:, 1]
    lgbm_proba = lgbm_new.predict_proba(X_val_red)[:, 1]
    end = time()

    # =======================================
    # WEIGHTED VOTING (STACKING META-LEARNER)
    # =======================================
    hard_start = time()

    X_meta_val = np.column_stack((cb_proba, xgb_proba, lgbm_proba))
    print(
        '=========================================\n'
        'Logistic Regression Meta-Learner Training\n'
        '========================================='
    )
    meta_model = LogisticRegression(random_state=42)
    meta_model.fit(X_meta_val, y_val)

    print("\nLearned Model Weights (Coefficients):")
    print(f"CatBoost Weight:  {meta_model.coef_[0][0]:.4f}")
    print(f"XGBoost Weight: {meta_model.coef_[0][1]:.4f}")
    print(f"LightGBM Weight: {meta_model.coef_[0][2]:.4f}")

    weighted_proba = meta_model.predict_proba(X_meta_val)[:, 1]

    hard_end = time()
    hard_inf_time = (end - start) + (hard_end - hard_start)
    print(f'Weighted Voting Inference Time: {hard_inf_time:.4f}')

    print(
        '\n'
        '========================================\n'
        'WEIGHTED ENSEMBLE EVALUATION ON TEST SET\n'
        '========================================'
    )
    # threshold = 0.5
    # for thr in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]:
    weighted_pred = (weighted_proba >= 0.15).astype(int)
    con_mat = confusion_matrix(y_val, weighted_pred)

    print(f'\n---Classification Report (threshold = {0.15})---')
    print(classification_report(y_val, weighted_pred))
    print(f'Accuracy Score: {accuracy_score(y_val, weighted_pred):.4f}')
    print(f'ROC AUC Score: {roc_auc_score(y_val, weighted_proba):.4f}')
    print(f'Confusion Matrix:\n {con_mat}')

    results = {
        "group_removed": group_name,
        "training_time": round(train_end - train_start, 4),
        "inference_time": round(hard_inf_time, 4),
        "evaluation": {
            "threshold": 0.15,
            "accuracy": round(accuracy_score(y_val, weighted_pred), 4),
            "roc_auc": round(roc_auc_score(y_val, weighted_proba), 4),
            "confusion_matrix": con_mat.tolist(),
            "classification_report": classification_report(y_val, weighted_pred, output_dict=True)
        },
        "model_params": {
            "catboost": cb_params,
            "xgboost": xgb_new.get_params(),
            "lgbm": lgbm_new.get_params()
        }
    }
    # Lưu models riêng
    model_dir = Path(f'ablation_results/wo_{group_name}')
    model_dir.mkdir(parents=True, exist_ok=True)

    cb_new.save_model(str(model_dir / 'catboost.cbm'))
    xgb_new.save_model(str(model_dir / 'xgboost.json'))
    joblib.dump(lgbm_new, model_dir / 'lgbm.joblib')
    meta_model.save_model(str(model_dir / 'meta_model.json'))

    output_path = model_dir / 'results.json'
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=4, default=str)
    print(f'Saved results to {output_path}')
    

Columns of noise_residual_extract: [2, 1, 3, 0, 33, 32]
Training(w/o noise_residual_extract)...
0:	test: 0.7084348	best: 0.7084348 (0)	total: 1.32s	remaining: 3h 40m 5s
200:	test: 0.7547369	best: 0.7547369 (200)	total: 12m 26s	remaining: 10h 6m 20s
400:	test: 0.7618500	best: 0.7618512 (399)	total: 27m 42s	remaining: 11h 3m 11s
600:	test: 0.7650699	best: 0.7650699 (600)	total: 42m 35s	remaining: 11h 5m 57s
800:	test: 0.7674103	best: 0.7674103 (800)	total: 56m 7s	remaining: 10h 44m 34s
1000:	test: 0.7689476	best: 0.7689476 (1000)	total: 1h 7m 26s	remaining: 10h 6m 16s
1200:	test: 0.7702394	best: 0.7702440 (1196)	total: 1h 16m 55s	remaining: 9h 23m 31s
1400:	test: 0.7709986	best: 0.7710370 (1385)	total: 1h 25m 49s	remaining: 8h 46m 48s
1600:	test: 0.7716444	best: 0.7716639 (1596)	total: 1h 34m 15s	remaining: 8h 14m 29s
1800:	test: 0.7721709	best: 0.7721709 (1800)	total: 1h 42m 38s	remaining: 7h 47m 17s
2000:	test: 0.7724826	best: 0.7725157 (1990)	total: 1h 50m 40s	remaining: 7h 22m 26s
22

d:\HOC_TREN_TRUONG\HOC_MAY_THONG_KE\DO_AN\FMaFE---Fusion-AI-Manipulation-Image-Feature-Extraction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[LightGBM] [Info] Number of positive: 619357, number of negative: 1798368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.253491 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 23970
[LightGBM] [Info] Number of data points in the train set: 2417725, number of used features: 94
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256173 -> initscore=-1.065953
[LightGBM] [Info] Start training from score -1.065953
Training Time(w/o noise_residual_extract): 10767.9759

Predicting...

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 174.9631

WEIGHTED ENSEMBLE EVALUATION ON VALID SET (W/O noise_residual_extract)

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    998590
           1       0.28      0.08      0.12     75257

    accuracy                           0.92   1073847
   macro avg       

d:\HOC_TREN_TRUONG\HOC_MAY_THONG_KE\DO_AN\FMaFE---Fusion-AI-Manipulation-Image-Feature-Extraction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[LightGBM] [Info] Number of positive: 619357, number of negative: 1798368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.274607 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 25245
[LightGBM] [Info] Number of data points in the train set: 2417725, number of used features: 99
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256173 -> initscore=-1.065953
[LightGBM] [Info] Start training from score -1.065953
Training Time(w/o fft_extract): 6252.8805

Predicting...

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 170.1825

WEIGHTED ENSEMBLE EVALUATION ON VALID SET (W/O fft_extract)

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.93      0.98      0.96    998590
           1       0.28      0.08      0.12     75257

    accuracy                           0.92   1073847
   macro avg       0.61      0.53      0.5

...

[LightGBM] [Info] Number of positive: 619357, number of negative: 1798368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.078922 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 24990
[LightGBM] [Info] Number of data points in the train set: 2417725, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256173 -> initscore=-1.065953
[LightGBM] [Info] Start training from score -1.065953
Training Time(w/o hprb_extract): 9304.7671

Predicting...

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 197.0319

WEIGHTED ENSEMBLE EVALUATION ON VALID SET (W/O hprb_extract)

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    998590
           1       0.29      0.08      0.12     75257

    accuracy                           0.92   1073847
   macro avg       0.61      0.53      0

d:\HOC_TREN_TRUONG\HOC_MAY_THONG_KE\DO_AN\FMaFE---Fusion-AI-Manipulation-Image-Feature-Extraction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[LightGBM] [Info] Number of positive: 619357, number of negative: 1798368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.195118 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7140
[LightGBM] [Info] Number of data points in the train set: 2417725, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256173 -> initscore=-1.065953
[LightGBM] [Info] Start training from score -1.065953
Training Time(w/o color_hist_extract): 5118.7846

Predicting...

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 202.3473

WEIGHTED ENSEMBLE EVALUATION ON VALID SET (W/O color_hist_extract)

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    998590
           1       0.31      0.05      0.08     75257

    accuracy                           0.93   1073847
   macro avg       0.62      

d:\HOC_TREN_TRUONG\HOC_MAY_THONG_KE\DO_AN\FMaFE---Fusion-AI-Manipulation-Image-Feature-Extraction\.venv\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[LightGBM] [Info] Number of positive: 619357, number of negative: 1798368
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.719531 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 24735
[LightGBM] [Info] Number of data points in the train set: 2417725, number of used features: 97
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256173 -> initscore=-1.065953
[LightGBM] [Info] Start training from score -1.065953
Training Time(w/o chroma_correlation_extract): 17016.7037

Predicting...

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 344.3646

WEIGHTED ENSEMBLE EVALUATION ON VALID SET (W/O chroma_correlation_extract)

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.93      0.99      0.96    998590
           1       0.27      0.06      0.10     75257

    accuracy                           0.92   1073847
   macro av

In [ ]:
for group_name, ids in feat_idx.items():
    local_path = Path(f'ablation_results/wo_{group_name}')
    group_cols = [col for col in X_train_df.columns if col in ids]
    if len(group_cols) <= 0:
        print(f"Not exist features of {group_name} in Top-100!")
        continue
    print(f"Columns of {group_name}: {group_cols}")

    # Remove columns
    X_val_red = X_val_df.drop(columns=group_cols)
    X_test_red = X_test_df.drop(columns=group_cols)
    
    # Load CatBoost Checkpoint
    if os.path.exists(local_path / 'catboost.cbm'):
        cb_new = CatBoostClassifier()
        cb_new.load_model(local_path / 'catboost.cbm')
    else:
        raise FileNotFoundError(f'CatBoost checkpoint does not exist at this path: "{local_path / 'catboost.cbm'}"')

    # Load XGBoost Checkpoint
    if os.path.exists(local_path / 'xgboost.json'):
        xgb_new = XGBClassifier()
        xgb_new.load_model(local_path / 'xgboost.json')
    else:
        raise FileNotFoundError(f'XGBoost checkpoint does not exist at this path: "{local_path / 'xgboost.json'}"')

    # Load LightGBM Checkpoint
    if os.path.exists(local_path / 'lgbm.joblib'):
        lgbm_new = joblib.load(local_path / 'lgbm.joblib')
    else:
        raise FileNotFoundError(f'LightGBM checkpoint does not exist at this path: "{local_path / 'lgbm.joblib'}"')

    print('\nModel Loaded Successfully!')

    # =======================================
    # WEIGHTED VOTING (STACKING META-LEARNER)
    # =======================================
    if os.path.exists(Path(local_path / 'meta_model.json')):
        meta_model = LogisticRegression()
        meta_model.load_model(Path(local_path / 'meta_model.json'))
    else:
        raise FileNotFoundError(f'Meta_model checkpoint does not exist at this path: "{Path(local_path / 'meta_model.json')}"')
    print('\nEnsembling (Weighted Voting)...')
    hard_start = time()
    cb_proba_test = cb_new.predict_proba(X_test_red)[:, 1]
    xgb_proba_test = xgb_new.predict_proba(X_test_red)[:, 1]
    lgbm_proba_test = lgbm_new.predict_proba(X_test_red)[:, 1]

    X_meta_test = np.column_stack((cb_proba_test, xgb_proba_test, lgbm_proba_test))
    weighted_proba = meta_model.predict_proba(X_meta_test)[:, 1]

    hard_end = time()
    hard_inf_time = (end - start) + (hard_end - hard_start)
    print(f'Weighted Voting Inference Time: {hard_inf_time:.4f}')
    print(
        '\n'
        '========================================\n'
        'WEIGHTED ENSEMBLE EVALUATION ON TEST SET\n'
        '========================================'
    )
    # threshold = 0.5
    # for thr in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]:
    weighted_pred = (weighted_proba >= 0.15).astype(int)
    con_mat = confusion_matrix(y_test, weighted_pred)

    print(f'\n---Classification Report (threshold = {0.15})---')
    print(classification_report(y_test, weighted_pred))
    print(f'Accuracy Score: {accuracy_score(y_test, weighted_pred):.4f}')
    print(f'ROC AUC Score: {roc_auc_score(y_test, weighted_proba):.4f}')
    print(f'Confusion Matrix:\n {con_mat}')
    test_results = {
        "inference_time": round(hard_inf_time, 4),
        "evaluation": {
            "threshold": 0.15,
            "accuracy": round(accuracy_score(y_test, weighted_pred), 4),
            "roc_auc": round(roc_auc_score(y_test, weighted_proba), 4),
            "confusion_matrix": con_mat.tolist(),
            "classification_report": classification_report(y_test, weighted_pred, output_dict=True)
        }
    }
    local_path.mkdir(parents=True, exist_ok=True)
    output_path = local_path / 'test_results.json'
    with open(output_path, 'w') as f:
        json.dump(test_results, f, indent=4, default=str)
    print(f'Saved test-results to {output_path}')

Columns of noise_residual_extract: [2, 1, 3, 0, 33, 32]

Model Loaded Successfully!
Logistic Regression Meta-Learner Training

Learned Model Weights (Coefficients):
CatBoost Weight:  1.9234
XGBoost Weight: 1.7658
LightGBM Weight: -0.8585

Ensembling (Weighted Voting)...
Weighted Voting Inference Time: 385.7465

WEIGHTED ENSEMBLE EVALUATION ON TEST SET

---Classification Report (threshold = 0.15)---
              precision    recall  f1-score   support

           0       0.94      0.98      0.96    981326
           1       0.21      0.06      0.10     67409

    accuracy                           0.92   1048735
   macro avg       0.57      0.52      0.53   1048735
weighted avg       0.89      0.92      0.91   1048735

Accuracy Score: 0.9243
ROC AUC Score: 0.7599
Confusion Matrix:
 [[965067  16259]
 [ 63089   4320]]
Saved results to ablation_results\wo_noise_residual_extract\test_results.json
Not exist features of wavelet_extract in Top-100!
Columns of fft_extract: [51]

Model Loaded S

In [ ]:
from abc import ABC, abstractmethod
class BaseFilterMethod(ABC):
    def __init__(self):
        self.score_dict = dict()

    @abstractmethod
    def score(
        self,
        X: np.ndarray,
        y: np.ndarray,
        indices: np.ndarray = None,
        bins: int = None
    ) -> dict[int, float]:
        ...

class OutOfCoreContingencyFilter(BaseFilterMethod):
    def _build_joint_counts(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray,
            bins: int,
            batch_size=100_000
        ):
        n_cols = X.shape[1]
        n_samples = len(indices) if indices is not None else X.shape[0]
        
        # 1. Map Y to discrete integer classes
        unique_y, y_mapped = np.unique(y, return_inverse=True)
        num_y = len(unique_y)
        
        # 2. Get global quantiles for X using a memory-safe random sample
        sample_size = min(1_000_000, n_samples)
        if indices is not None:
            sample_idx = np.random.choice(indices, size=sample_size, replace=False)
        else:
            sample_idx = np.random.choice(n_samples, size=sample_size, replace=False)
        sample_idx.sort()
        
        sample_buffer = []
        for i in range(0, sample_size, batch_size):
            sample_buffer.append(X[sample_idx[i:i+batch_size]])
        X_sample = np.vstack(sample_buffer)
        
        q = np.linspace(0, 100, bins + 1)
        quantiles_raw = np.percentile(X_sample, q, axis=0)
        
        # Handle columns with duplicate percentiles (e.g. constant columns)
        quantiles_list = []
        for c in range(n_cols):
            quantiles_list.append(np.unique(quantiles_raw[:, c]))
            
        # 3. Accumulate joint counts (X_binned, Y)
        num_x_bins = bins + 2 
        joint_counts = np.zeros((n_cols, num_x_bins, num_y), dtype=np.int64)
        loop_indices = indices if indices is not None else np.arange(n_samples)
        
        # Slice the array via rows
        for i in tqdm(range(0, n_samples, batch_size), desc='Reading Chunks', leave=False):
            batch_idx = loop_indices[i:i+batch_size]
            X_chunk = X[batch_idx]
            y_chunk_mapped = y_mapped[i:i+batch_size]
            
            for c in range(n_cols):
                # Bin the floats into discrete integers directly in RAM
                x_binned = np.digitize(X_chunk[:, c], quantiles_list[c])
                
                # Fast 2D frequency accumulation
                flat_indices = x_binned * num_y + y_chunk_mapped
                counts = np.bincount(flat_indices, minlength=num_x_bins * num_y)
                joint_counts[c] += counts.reshape(num_x_bins, num_y)
                
        return joint_counts

    def _compute_entropies(self, joint_counts):
        eps = 1e-9
        
        total = np.sum(joint_counts, axis=(1, 2), keepdims=True)
        P_xy = joint_counts / total
        
        P_x = np.sum(P_xy, axis=2) 
        P_y = np.sum(P_xy, axis=1) 
        
        H_x = -np.sum(P_x * np.log2(P_x + eps), axis=1)
        H_y = -np.sum(P_y * np.log2(P_y + eps), axis=1)
        H_xy = -np.sum(P_xy * np.log2(P_xy + eps), axis=(1, 2))
        
        return H_x, H_y, H_xy

class InformationGain(OutOfCoreContingencyFilter):
    def score(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            bins: int = 10
        ):
        joint_counts = self._build_joint_counts(X, y, indices, bins)
        H_x, H_y, H_xy = self._compute_entropies(joint_counts)
        
        IG = H_x + H_y - H_xy
        
        for c in range(X.shape[1]):
            self.score_dict[c] = IG[c]
            
        return dict(sorted(self.score_dict.items(), key=lambda item: item[1], reverse=True))

class SymmetricUncertainty(OutOfCoreContingencyFilter):
    def score(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            bins: int = 10
        ):
        joint_counts = self._build_joint_counts(X, y, indices, bins)
        H_x, H_y, H_xy = self._compute_entropies(joint_counts)
        
        IG = H_x + H_y - H_xy
        SU = 2 * IG / (H_x + H_y + 1e-9)
        
        for c in range(X.shape[1]):
            self.score_dict[c] = SU[c]
            
        return dict(sorted(self.score_dict.items(), key=lambda item: item[1], reverse=True))

class BaseScoreCombiner(ABC):
    @abstractmethod
    def combine(self, scores: list[dict[int, float]]) -> list[int]:
        ...

    def _cut(self, scores: dict[int, float]) -> list[int]:
        if getattr(self, 'top_k', None):
            sorted_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:self.top_k]
            top_keys   = [k for k, v in sorted_items]
            top_values = [v for k, v in sorted_items]
            return top_keys, top_values
    
        if getattr(self, 'threshold', None):
            return [k for k, v in scores.items() if v >= self.threshold]
        
        raise ValueError('Specify either top_k or threshold.')

class MeanCombiner(BaseScoreCombiner):
    def __init__(
            self,
            top_k: int = 1,
            threshold: float = None
        ):
        self.top_k = top_k
        self.threshold = threshold
    
    def combine(
            self,
            scores: list[dict[int, float]]
        ):
        keys = scores[0].keys()
        merged = {k: np.mean([s[k] for s in scores]) for k in keys}
        
        return self._cut(merged)
    
class IntersectCombiner(BaseScoreCombiner):
    def __init__(
            self,
            top_k: int = 1,
            min_agreement: int = 1
        ):
        self.top_k = top_k
        self.min_agreement = min_agreement

    def combine(
            self,
            scores: list[dict[int, float]]
        ):
        min_agree = self.min_agreement or len(scores)

        def top_keys(s):
            return set(sorted(s, key=s.get, reverse=True)[:self.top_k])
        
        agreement = {
            k: sum(k in top_keys(s) for s in scores)
            for k in scores[0].keys()
        }
        
        return [k for k, count in agreement.items() if count >= min_agree]
    
class Filter:
    def __init__(
            self,
            methods: list[BaseFilterMethod],
            combiner: BaseScoreCombiner = None
        ):
        self.methods = methods
        self.combiner = combiner
        self.selected_columns = None
        self.importances = None
    
    def fit(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None,
            n_bins: int = 10
        ):
        print('Start calculating feature scores...')
        
        scores = []
        for i, m in enumerate(self.methods):
            print(f"\n-> Running Filter Method {i+1}/{len(self.methods)}: {m.__class__.__name__}")
            scores.append(m.score(X, y, indices, bins=n_bins))
            
        print('\nChoosing columns to keep...')
        self.selected_columns, self.importances = (
            list(scores[0].keys())
            if len(scores) == 1 and not self.combiner
            else self.combiner.combine(scores)
        )
        return self
    
    def transform(
            self,
            X: np.ndarray
        ):
        if self.selected_columns is None:
            raise RuntimeError("Call fit before transform")
        
        return X[:, self.selected_columns]
    
    def fit_transform(
            self,
            X: np.ndarray,
            y: np.ndarray,
            indices: np.ndarray = None
        ):
        self.fit(X, y, indices)
        
        return self.transform(X)

In [28]:
from tqdm.auto import tqdm

top_k = 100
ig = InformationGain()
su = SymmetricUncertainty()
combiner = MeanCombiner(top_k=top_k)

print(f'\nFinding top {top_k} features...')
feature_filter = Filter(methods=[ig, su], combiner=combiner)
feature_filter.fit(X=X_train, y=y_train, n_bins=30)

print(type(feature_filter.selected_columns))


Finding top 100 features...
Start calculating feature scores...

-> Running Filter Method 1/2: InformationGain



-> Running Filter Method 2/2: SymmetricUncertainty



Choosing columns to keep...
<class 'list'>
